In [1]:
!pip install -q transformers accelerate datasets sqlparse tqdm


In [2]:
from pathlib import Path
import zipfile

zip_path = Path("spider_data.zip")
extract_root = Path("spider_data")

print("Zip exists:", zip_path.exists())

if not extract_root.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_root)
    print("Extracted to", extract_root.resolve())
else:
    print("Already extracted at", extract_root.resolve())

# Find dev.json and tables.json inside the extracted folder
dev_json_candidates = list(extract_root.rglob("dev.json"))
tables_json_candidates = list(extract_root.rglob("tables.json"))
db_dir_candidates = list(extract_root.rglob("database"))

print("Found dev.json:", dev_json_candidates)
print("Found tables.json:", tables_json_candidates)
print("Found database dirs:", db_dir_candidates)


Zip exists: True
Extracted to /teamspace/studios/this_studio/spider_data
Found dev.json: [PosixPath('spider_data/spider_data/dev.json')]
Found tables.json: [PosixPath('spider_data/spider_data/tables.json')]
Found database dirs: [PosixPath('spider_data/spider_data/database'), PosixPath('spider_data/__MACOSX/spider_data/database')]


In [3]:
import json
from pathlib import Path

# Use the candidates from previous cell
extract_root = Path("spider_data")

dev_json_candidates = list(extract_root.rglob("dev.json"))
tables_json_candidates = list(extract_root.rglob("tables.json"))
db_dir_candidates = list(extract_root.rglob("database"))

assert dev_json_candidates, "No dev.json found inside spider_data."
assert tables_json_candidates, "No tables.json found inside spider_data."
assert db_dir_candidates, "No 'database' directory found inside spider_data."

# Pick the first dev.json and use its parent as SPIDER_ROOT
dev_json_path = dev_json_candidates[0]
SPIDER_ROOT = dev_json_path.parent
TABLES_PATH = tables_json_candidates[0]
DATABASE_DIR = db_dir_candidates[0]

print("SPIDER_ROOT:", SPIDER_ROOT)
print("DEV JSON:", dev_json_path)
print("TABLES:", TABLES_PATH)
print("DATABASE DIR:", DATABASE_DIR)

# Load dev set (Spider official dev.json is a list of dicts)
with open(dev_json_path, "r", encoding="utf-8") as f:
    spider_dev = json.load(f)

print("Number of dev examples:", len(spider_dev))
print(spider_dev[0].keys())
spider_dev[0]


SPIDER_ROOT: spider_data/spider_data
DEV JSON: spider_data/spider_data/dev.json
TABLES: spider_data/spider_data/tables.json
DATABASE DIR: spider_data/spider_data/database
Number of dev examples: 1034
dict_keys(['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql'])


{'db_id': 'concert_singer',
 'query': 'SELECT count(*) FROM singer',
 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'singer'],
 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'singer'],
 'question': 'How many singers do we have?',
 'question_toks': ['How', 'many', 'singers', 'do', 'we', 'have', '?'],
 'sql': {'from': {'table_units': [['table_unit', 1]], 'conds': []},
  'select': [False, [[3, [0, [0, 0, False], None]]]],
  'where': [],
  'groupBy': [],
  'having': [],
  'orderBy': [],
  'limit': None,
  'intersect': None,
  'union': None,
  'except': None}}

In [4]:
with open(TABLES_PATH, "r", encoding="utf-8") as f:
    spider_tables = json.load(f)

db_schemas = {}

for db in spider_tables:
    db_id = db["db_id"]
    tables = db["table_names_original"]          # list of table names
    columns = db["column_names_original"]        # list of [table_id, column_name]

    # collect columns per table
    table_cols = {i: [] for i in range(len(tables))}
    for table_id, col_name in columns:
        if table_id == -1:  # '*' pseudo table
            continue
        table_cols[table_id].append(col_name)

    schema_strings = []
    for i, tname in enumerate(tables):
        cols = table_cols[i]
        cols_str = ", ".join(cols)
        schema_strings.append(f"{tname}({cols_str})")

    db_schemas[db_id] = " | ".join(schema_strings)

# Sanity check a few schemas
list(db_schemas.items())[:3]


[('perpetrator',
  'perpetrator(Perpetrator_ID, People_ID, Date, Year, Location, Country, Killed, Injured) | people(People_ID, Name, Height, Weight, Home Town)'),
 ('college_2',
  'classroom(building, room_number, capacity) | department(dept_name, building, budget) | course(course_id, title, dept_name, credits) | instructor(ID, name, dept_name, salary) | section(course_id, sec_id, semester, year, building, room_number, time_slot_id) | teaches(ID, course_id, sec_id, semester, year) | student(ID, name, dept_name, tot_cred) | takes(ID, course_id, sec_id, semester, year, grade) | advisor(s_ID, i_ID) | time_slot(time_slot_id, day, start_hr, start_min, end_hr, end_min) | prereq(course_id, prereq_id)'),
 ('flight_company',
  'airport(id, City, Country, IATA, ICAO, name) | operate_company(id, name, Type, Principal_activities, Incorporated_in, Group_Equity_Shareholding) | flight(id, Vehicle_Flight_number, Date, Pilot, Velocity, Altitude, airport_id, company_id)')]

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # change here if you use another

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

DEVICE = "cuda"
MAX_NEW_TOKENS = 256


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
SYSTEM_PROMPT = (
    "You are a text-to-SQL model. "
    "Given a natural language question and the database schema, "
    "output ONLY the SQL query that answers the question. No explanation."
)

def build_chat_messages(question: str, db_id: str):
    schema = db_schemas.get(db_id, "")
    user_content = (
        f"Database schema:\n{schema}\n\n"
        f"Question: {question}\n\n"
        f"Write the SQL query."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    return messages


In [14]:
import re
import torch

def clean_sql_output(text: str) -> str:
    text = text.strip()

    # 1) Strip Markdown fences like ```sql ... ```
    if text.startswith("```"):
        # Remove opening ```... line
        text = re.sub(r"^```[a-zA-Z0-9_]*\s*", "", text)
        # Remove trailing ```
        text = re.sub(r"\s*```$", "", text)
        text = text.strip()

    # 2) Remove leading labels like "SQL:" / "Here is the SQL:"
    prefixes = [
        "sql:", "SQL:", "Sql:",
        "here is the sql query:", "here is the sql:", "here is the query:",
        "the sql query is:", "the query is:"
    ]
    lower = text.lower()
    for p in prefixes:
        if lower.startswith(p.lower()):
            text = text[len(p):].strip()
            lower = text.lower()
            break

    # 3) Try to extract from first "select" to first semicolon
    lower = text.lower()
    sel_idx = lower.find("select")
    if sel_idx != -1:
        text = text[sel_idx:]  # from SELECT...
    text = text.strip()

    # Cut at first semicolon if present (assume query ends there)
    if ";" in text:
        text = text.split(";", 1)[0] + ";"

    # Otherwise just take first line
    else:
        text = text.splitlines()[0].strip()

    # Final collapse of whitespace
    return " ".join(text.split())

def generate_sql_for_example(example):
    question = example["question"]
    db_id = example["db_id"]

    messages = build_chat_messages(question, db_id)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    sql = clean_sql_output(raw_text)
    return sql




In [15]:
import random

def debug_generate_sql_for_example(example):
    question = example["question"]
    db_id = example["db_id"]

    messages = build_chat_messages(question, db_id)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    cleaned = clean_sql_output(raw_text)
    return raw_text, cleaned

# Inspect 5 random dev examples
indices = random.sample(range(len(spider_dev)), 5)
for idx in indices:
    ex = spider_dev[idx]
    raw, cleaned = debug_generate_sql_for_example(ex)
    print(f"=== Example {idx} ===")
    print("Q:", ex["question"])
    print("DB:", ex["db_id"])
    print("GOLD:", ex["query"])
    print("\nRAW MODEL OUTPUT:\n", raw)
    print("\nCLEANED SQL:\n", cleaned)
    print("---------------\n")




=== Example 769 ===
Q: What is the official language used in the country the name of whose head of state is Beatrix.
DB: world_1
GOLD: SELECT T2.Language FROM country AS T1 JOIN countrylanguage AS T2 ON T1.Code  =  T2.CountryCode WHERE T1.HeadOfState  =  "Beatrix" AND T2.IsOfficial  =  "T"

RAW MODEL OUTPUT:
 ```sql
SELECT T1.Language FROM city AS T1 INNER JOIN country AS T2 ON T1.CountryCode = T2.Code WHERE T2.HeadOfState = 'Beatrix';
```

CLEANED SQL:
 SELECT T1.Language FROM city AS T1 INNER JOIN country AS T2 ON T1.CountryCode = T2.Code WHERE T2.HeadOfState = 'Beatrix';
---------------

=== Example 156 ===
Q: What is the horsepower of the car with the greatest accelerate?
DB: car_1
GOLD: SELECT T1.horsepower FROM CARS_DATA AS T1 ORDER BY T1.accelerate DESC LIMIT 1;

RAW MODEL OUTPUT:
 ```sql
SELECT T1.Horsepower FROM cars_data AS T1 INNER JOIN car_names AS T2 ON T1.MakeId = T2.MakeId WHERE T1.Accelerate = ( SELECT MAX(Accelerate) FROM cars_data );
```

CLEANED SQL:
 SELECT T1.Horse

In [16]:
from pathlib import Path
from tqdm.auto import tqdm
import sqlparse

DRY_RUN_LIMIT = 20  # small number for test

def normalize_sql(sql: str) -> str:
    if sql is None:
        return ""
    try:
        formatted = sqlparse.format(
            sql,
            keyword_case="lower",
            identifier_case="lower",
            strip_comments=True,
        )
    except Exception:
        formatted = sql
    return " ".join(formatted.split())

gold_file = Path("dev_gold_head.sql")
pred_file = Path("dev_pred_head.sql")

preds = []
golds = []

with gold_file.open("w", encoding="utf-8") as fg, pred_file.open("w", encoding="utf-8") as fp:
    for i, ex in enumerate(tqdm(spider_dev[:DRY_RUN_LIMIT], desc="Dry run")):
        gold_sql = ex["query"].strip().replace("\n", " ")
        db_id = ex["db_id"]

        pred_sql = generate_sql_for_example(ex).strip().replace("\n", " ")

        fg.write(f"{gold_sql}\t{db_id}\n")
        fp.write(pred_sql + "\n")

        golds.append(normalize_sql(gold_sql))
        preds.append(normalize_sql(pred_sql))

print("Wrote:", gold_file.resolve())
print("Wrote:", pred_file.resolve())

correct = sum(p == g for p, g in zip(preds, golds))
em = correct / len(golds) * 100
print(f"Dry-run EM (normalized, rough): {em:.2f}% ({correct}/{len(golds)})")


Dry run:   0%|          | 0/20 [00:00<?, ?it/s]

Wrote: /teamspace/studios/this_studio/dev_gold_head.sql
Wrote: /teamspace/studios/this_studio/dev_pred_head.sql
Dry-run EM (normalized, rough): 0.00% (0/20)


In [17]:
print("dev_gold_head.sql:")
print(Path("dev_gold_head.sql").read_text(encoding="utf-8"))

print("\n---\ndev_pred_head.sql:")
print(Path("dev_pred_head.sql").read_text(encoding="utf-8"))


dev_gold_head.sql:
SELECT count(*) FROM singer	concert_singer
SELECT count(*) FROM singer	concert_singer
SELECT name ,  country ,  age FROM singer ORDER BY age DESC	concert_singer
SELECT name ,  country ,  age FROM singer ORDER BY age DESC	concert_singer
SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'	concert_singer
SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'	concert_singer
SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1	concert_singer
SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1	concert_singer
SELECT DISTINCT country FROM singer WHERE age  >  20	concert_singer
SELECT DISTINCT country FROM singer WHERE age  >  20	concert_singer
SELECT country ,  count(*) FROM singer GROUP BY country	concert_singer
SELECT country ,  count(*) FROM singer GROUP BY country	concert_singer
SELECT song_name FROM singer WHERE age  >  (SELECT avg(age) FROM singer)	concert_singer
SELECT song_name FR

In [18]:
from pathlib import Path
from tqdm.auto import tqdm
import sqlparse

def normalize_sql(sql: str) -> str:
    if sql is None:
        return ""
    try:
        formatted = sqlparse.format(
            sql,
            keyword_case="lower",
            identifier_case="lower",
            strip_comments=True,
        )
    except Exception:
        formatted = sql
    return " ".join(formatted.split())

gold_file = Path("dev_gold.sql")
pred_file = Path("dev_pred.sql")

preds = []
golds = []

with gold_file.open("w", encoding="utf-8") as fg, pred_file.open("w", encoding="utf-8") as fp:
    for ex in tqdm(spider_dev, desc="Full Spider dev"):
        gold_sql = ex["query"].strip().replace("\n", " ")
        db_id = ex["db_id"]

        pred_sql = generate_sql_for_example(ex).strip().replace("\n", " ")

        fg.write(f"{gold_sql}\t{db_id}\n")
        fp.write(pred_sql + "\n")

        golds.append(normalize_sql(gold_sql))
        preds.append(normalize_sql(pred_sql))

print("Wrote:", gold_file.resolve())
print("Wrote:", pred_file.resolve())

correct = sum(p == g for p, g in zip(preds, golds))
em = correct / len(golds) * 100
print(f"Full dev EM (normalized, rough): {em:.2f}% ({correct}/{len(golds)})")


Full Spider dev:   0%|          | 0/1034 [00:00<?, ?it/s]

Wrote: /teamspace/studios/this_studio/dev_gold.sql
Wrote: /teamspace/studios/this_studio/dev_pred.sql
Full dev EM (normalized, rough): 1.35% (14/1034)


In [19]:
from pathlib import Path

# Assuming you're running from THIS_STUDIO (where main.ipynb is)
SPIDER_ROOT = Path("spider_data/spider_data")
DATABASE_DIR = SPIDER_ROOT / "database"
TABLES_PATH = SPIDER_ROOT / "tables.json"

print("SPIDER_ROOT:", SPIDER_ROOT.resolve())
print("DATABASE_DIR:", DATABASE_DIR.resolve())
print("TABLES_PATH:", TABLES_PATH.resolve())

# Gold & pred we just created for dev
GOLD_FILE = Path("dev_gold.sql")
PRED_FILE = Path("dev_pred.sql")

print("GOLD_FILE:", GOLD_FILE.resolve())
print("PRED_FILE:", PRED_FILE.resolve())


SPIDER_ROOT: /teamspace/studios/this_studio/spider_data/spider_data
DATABASE_DIR: /teamspace/studios/this_studio/spider_data/spider_data/database
TABLES_PATH: /teamspace/studios/this_studio/spider_data/spider_data/tables.json
GOLD_FILE: /teamspace/studios/this_studio/dev_gold.sql
PRED_FILE: /teamspace/studios/this_studio/dev_pred.sql


In [40]:
import nltk, os
from pathlib import Path

nltk_data_dir = Path("nltk_punkt_data")
nltk_data_dir.mkdir(exist_ok=True)

nltk.download("punkt", download_dir=str(nltk_data_dir))

# Also sometimes needed:
nltk.download("punkt_tab", download_dir=str(nltk_data_dir))

print("Downloaded punkt to:", nltk_data_dir.resolve())


[nltk_data] Downloading package punkt to nltk_punkt_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to nltk_punkt_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Downloaded punkt to: /teamspace/studios/this_studio/nltk_punkt_data


In [41]:
import subprocess, sys, os
from pathlib import Path

eval_script = "spider_official_eval/evaluation.py"

env = os.environ.copy()
env["NLTK_DATA"] = str(Path("nltk_punkt_data").resolve())

cmd = [
    sys.executable,
    eval_script,
    "--gold", str(GOLD_FILE),
    "--pred", str(PRED_FILE),
    "--etype", "all",
    "--db", str(DATABASE_DIR),
    "--table", str(TABLES_PATH),
]

print("Running Spider evaluation...")
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env=env,        # <- THE FIX
)

stdout, stderr = proc.communicate()

print("=== Evaluation output ===")
print(stdout)
print("=== STDERR ===")
print(stderr)


Running Spider evaluation...
=== Evaluation output ===
easy pred: SELECT COUNT(DISTINCT Singer_ID) FROM singer;
easy gold: SELECT count(*) FROM singer

eval_err_num:1
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer);
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:2
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer);
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:3
easy pred: SELECT DISTINCT T1.Country FROM singer AS T1 INNER JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age > 20;
easy gold: SELECT DISTINCT country FROM singer WHERE age  >  20

eval_err_num:4
easy pred: SELECT DISTINCT T1.Country FROM singer AS T1 INNER JOIN singer_in_concert AS T2 ON 

In [42]:
print(stdout)


easy pred: SELECT COUNT(DISTINCT Singer_ID) FROM singer;
easy gold: SELECT count(*) FROM singer

eval_err_num:1
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer);
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:2
medium pred: SELECT T2.Name, T1.Song_Release_year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer);
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:3
easy pred: SELECT DISTINCT T1.Country FROM singer AS T1 INNER JOIN concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T1.Age > 20;
easy gold: SELECT DISTINCT country FROM singer WHERE age  >  20

eval_err_num:4
easy pred: SELECT DISTINCT T1.Country FROM singer AS T1 INNER JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID WHERE T2.Age > 20;
easy gol